In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = "cuda" if torch.cuda.is_available() else "cpu"

def load(model_name):
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
    return tok, model

# Load models
hi_en_tok, hi_en_model = load("Helsinki-NLP/opus-mt-hi-en")
en_bho_tok, en_bho_model = load("nilayshenai/BART-English-to-Bhojpuri-Alpha1")

def translate(texts, tok, model, max_len=256, batch_size=16):
    results = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tok(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(device)

        outputs = model.generate(**inputs, max_length=max_len)
        decoded = tok.batch_decode(outputs, skip_special_tokens=True)
        results.extend(decoded)
    return results

# 📂 Read Excel
df = pd.read_excel("/content/test.xlsx")

# Column containing Hindi text
hindi_sentences = df["Hindi"].astype(str).tolist()

# Step 1: Hindi → English
english = translate(hindi_sentences, hi_en_tok, hi_en_model)

# Step 2: English → Bhojpuri
bhojpuri = translate(english, en_bho_tok, en_bho_model)

# Add new column
df["bhojpuri"] = bhojpuri

# 💾 Save output
df.to_excel("output_bhojpuri.xlsx", index=False)

print("✅ Translation complete. Saved as output_bhojpuri.xlsx")


/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


✅ Translation complete. Saved as output_bhojpuri.xlsx
